# 02 — Analysis

Statistical benchmark scaffolding per source, an SMAP-vs-ISMN cross-validation setup, and a
**real** FAO-56 Penman-Monteith ET0 calculation (Allen et al. 1998). The weather inputs below
are simulated; the ET0 formula itself is the actual standard equation and produces correct
numbers for whatever inputs it is given.

See `docs/fao56-calculations.md` for the full derivation and notation this notebook follows
(that document may be produced by a parallel effort; the formula here is implemented directly
from Allen et al. 1998 regardless).


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
WINDOW_DAYS = 30
dates = pd.date_range("2025-06-01", periods=WINDOW_DAYS, freq="D", tz="UTC")

def seasonal_signal(n, base, amplitude, noise_std, period=365, phase=150, rng=rng):
    t = np.arange(n)
    signal = base + amplitude * np.sin(2 * np.pi * (t + phase) / period)
    return signal + rng.normal(0, noise_std, size=n)


## Per-source basic statistics and simulated missing-data rate

In [2]:
ndvi = seasonal_signal(WINDOW_DAYS, base=0.55, amplitude=0.15, noise_std=0.02)
cloud_mask = rng.random(WINDOW_DAYS) < 0.35
ndvi_observed = np.where(cloud_mask, np.nan, ndvi)

s1_vv_db = seasonal_signal(WINDOW_DAYS, base=-11.0, amplitude=1.5, noise_std=0.4)
layover_mask = rng.random(WINDOW_DAYS) < 0.05
s1_observed = np.where(layover_mask, np.nan, s1_vv_db)

basic_stats = pd.DataFrame({
    "source": ["Sentinel-2 NDVI", "Sentinel-1 gamma0_vv"],
    "n_expected": [WINDOW_DAYS, WINDOW_DAYS],
    "n_missing": [np.isnan(ndvi_observed).sum(), np.isnan(s1_observed).sum()],
    "missing_rate": [cloud_mask.mean(), layover_mask.mean()],
    "mean": [np.nanmean(ndvi_observed), np.nanmean(s1_observed)],
    "std": [np.nanstd(ndvi_observed), np.nanstd(s1_observed)],
})
basic_stats


,source,n_expected,n_missing,missing_rate,mean,std
0,Sentinel-2 NDVI,30,11,0.366667,0.581093,0.029067
1,Sentinel-1 gamma0_vv,30,4,0.133333,-10.619694,0.353854


## SMAP vs ISMN cross-validation

The user explicitly wants two independent soil-moisture pipelines (SMAP satellite, ISMN
ground stations) that validate each other rather than one blended series. This section
implements the real comparison statistics — correlation, RMSE, bias — against simulated
input series (the math is real, the inputs are placeholders).


In [3]:
smap_theta = np.clip(seasonal_signal(WINDOW_DAYS, base=0.22, amplitude=0.08, noise_std=0.015), 0.0, 1.0)
ismn_theta = np.clip(smap_theta + rng.normal(0.01, 0.02, size=WINDOW_DAYS), 0.0, 1.0)

def cross_validate(reference, candidate):
    """Real validation math: Pearson r, RMSE, mean bias. Inputs may be simulated; this
    function is what notebook 04 also reuses on live data once credentials exist."""
    reference = np.asarray(reference, dtype=float)
    candidate = np.asarray(candidate, dtype=float)
    mask = ~(np.isnan(reference) | np.isnan(candidate))
    ref, cand = reference[mask], candidate[mask]
    if ref.size < 2:
        raise ValueError("Not enough paired observations to cross-validate")
    r = np.corrcoef(ref, cand)[0, 1]
    rmse = np.sqrt(np.mean((cand - ref) ** 2))
    bias = np.mean(cand - ref)
    return {"n_pairs": int(ref.size), "pearson_r": r, "rmse_m3m3": rmse, "bias_m3m3": bias}

smap_vs_ismn = cross_validate(reference=ismn_theta, candidate=smap_theta)
smap_vs_ismn


{'n_pairs': 30,
 'pearson_r': np.float64(0.7402068623588531),
 'rmse_m3m3': np.float64(0.016928972952036835),
 'bias_m3m3': np.float64(-0.004530670873911579)}

## FAO-56 Penman-Monteith ET0 — real calculation

Full implementation of the FAO-56 reference evapotranspiration equation (Allen et al. 1998,
Chapter 4, Eq. 6), applied to a simulated daily weather dataframe. Every intermediate term
(saturation vapor pressure, actual vapor pressure, slope of the vapor pressure curve,
psychrometric constant, net radiation) is computed explicitly and matches the notation in
`docs/fao56-calculations.md`.


In [4]:
def fao56_et0(t_mean_c, t_max_c, t_min_c, rh_mean_pct, wind_2m_ms, solar_rad_mj_m2_day,
               elevation_m, latitude_deg, day_of_year):
    """FAO-56 Penman-Monteith reference evapotranspiration, mm/day.

    Implements Allen, Pereira, Raes & Smith (1998), FAO Irrigation and Drainage Paper 56,
    Chapter 4, Equation 6. All inputs are scalars or equal-length arrays (daily time step).
    """
    t_mean_c = np.asarray(t_mean_c, dtype=float)
    t_max_c = np.asarray(t_max_c, dtype=float)
    t_min_c = np.asarray(t_min_c, dtype=float)
    rh_mean_pct = np.asarray(rh_mean_pct, dtype=float)
    wind_2m_ms = np.asarray(wind_2m_ms, dtype=float)
    solar_rad_mj_m2_day = np.asarray(solar_rad_mj_m2_day, dtype=float)

    # Atmospheric pressure (kPa) from elevation — FAO-56 Eq. 7
    P = 101.3 * ((293 - 0.0065 * elevation_m) / 293) ** 5.26

    # Psychrometric constant (kPa/degC) — FAO-56 Eq. 8
    gamma = 0.000665 * P

    # Saturation vapor pressure (kPa) at Tmax and Tmin — FAO-56 Eq. 11
    def e_sat(t):
        return 0.6108 * np.exp((17.27 * t) / (t + 237.3))

    es = (e_sat(t_max_c) + e_sat(t_min_c)) / 2  # FAO-56 Eq. 12

    # Actual vapor pressure from mean RH — FAO-56 Eq. 19 (simplified form)
    ea = (rh_mean_pct / 100.0) * es

    # Slope of saturation vapor pressure curve at Tmean (kPa/degC) — FAO-56 Eq. 13
    delta = (4098 * e_sat(t_mean_c)) / (t_mean_c + 237.3) ** 2

    # Extraterrestrial radiation Ra (MJ/m2/day) — FAO-56 Eq. 21-25
    lat_rad = np.deg2rad(latitude_deg)
    dr = 1 + 0.033 * np.cos(2 * np.pi * day_of_year / 365)  # inverse relative Earth-Sun distance
    decl = 0.409 * np.sin(2 * np.pi * day_of_year / 365 - 1.39)  # solar declination
    ws = np.arccos(np.clip(-np.tan(lat_rad) * np.tan(decl), -1, 1))  # sunset hour angle
    Gsc = 0.0820  # solar constant, MJ/m2/min
    Ra = (24 * 60 / np.pi) * Gsc * dr * (
        ws * np.sin(lat_rad) * np.sin(decl) + np.cos(lat_rad) * np.cos(decl) * np.sin(ws)
    )

    # Clear-sky radiation Rso (MJ/m2/day) — FAO-56 Eq. 37 (simplified, no turbidity data)
    Rso = (0.75 + 2e-5 * elevation_m) * Ra

    # Net shortwave radiation, albedo=0.23 for the reference crop — FAO-56 Eq. 38
    albedo = 0.23
    Rns = (1 - albedo) * solar_rad_mj_m2_day

    # Net longwave radiation — FAO-56 Eq. 39
    sigma = 4.903e-9  # Stefan-Boltzmann constant, MJ K^-4 m^-2 day^-1
    t_max_k = t_max_c + 273.16
    t_min_k = t_min_c + 273.16
    Rnl = sigma * ((t_max_k**4 + t_min_k**4) / 2) * (0.34 - 0.14 * np.sqrt(np.clip(ea, 0, None))) * (
        1.35 * np.clip(solar_rad_mj_m2_day / np.clip(Rso, 1e-6, None), 0.05, 1.0) - 0.35
    )

    Rn = Rns - Rnl  # net radiation, FAO-56 Eq. 40

    G = 0.0  # soil heat flux, ~0 for daily time step — FAO-56 Eq. 42

    # FAO-56 Penman-Monteith, Eq. 6
    numerator = 0.408 * delta * (Rn - G) + gamma * (900 / (t_mean_c + 273)) * wind_2m_ms * (es - ea)
    denominator = delta + gamma * (1 + 0.34 * wind_2m_ms)
    et0 = numerator / denominator

    return {
        "et0_mm_day": et0, "Rn_mj_m2_day": Rn, "Ra_mj_m2_day": Ra, "Rso_mj_m2_day": Rso,
        "delta_kpa_c": delta, "gamma_kpa_c": gamma, "es_kpa": es, "ea_kpa": ea,
    }


In [5]:
# Apply to a simulated Aegean (İzmir/Manisa) daily weather dataframe — operational line
weather_df = pd.DataFrame({
    "date": dates,
    "t_mean_c": seasonal_signal(WINDOW_DAYS, base=24, amplitude=4, noise_std=1.0),
    "t_max_c": seasonal_signal(WINDOW_DAYS, base=30, amplitude=4, noise_std=1.2),
    "t_min_c": seasonal_signal(WINDOW_DAYS, base=18, amplitude=3, noise_std=1.0),
    "rh_mean_pct": np.clip(seasonal_signal(WINDOW_DAYS, base=55, amplitude=10, noise_std=3), 0, 100),
    "wind_2m_ms": np.abs(rng.normal(2.2, 0.6, WINDOW_DAYS)),
    "solar_rad_mj_m2_day": np.clip(seasonal_signal(WINDOW_DAYS, base=24, amplitude=4, noise_std=1.5), 0, None),
})

ELEVATION_M = 25.0       # placeholder — Aegean coastal plain
LATITUDE_DEG = 38.55     # placeholder AOI centroid

result = fao56_et0(
    t_mean_c=weather_df["t_mean_c"], t_max_c=weather_df["t_max_c"], t_min_c=weather_df["t_min_c"],
    rh_mean_pct=weather_df["rh_mean_pct"], wind_2m_ms=weather_df["wind_2m_ms"],
    solar_rad_mj_m2_day=weather_df["solar_rad_mj_m2_day"],
    elevation_m=ELEVATION_M, latitude_deg=LATITUDE_DEG,
    day_of_year=weather_df["date"].dt.dayofyear.values,
)
weather_df["et0_mm_day"] = result["et0_mm_day"]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(weather_df["date"], weather_df["et0_mm_day"], marker="o", ms=3, color="tab:red")
ax.set_title("FAO-56 Penman-Monteith ET0 (real calc, simulated weather inputs)")
ax.set_ylabel("ET0 (mm/day)")
plt.tight_layout()
plt.show()
weather_df[["date", "t_mean_c", "solar_rad_mj_m2_day", "et0_mm_day"]].head()


/var/folders/0s/rggf_js519j8hsgy53fr_jx80000gn/T/ipykernel_76148/2264021151.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,date,t_mean_c,solar_rad_mj_m2_day,et0_mm_day
0,2025-06-01 00:00:00+00:00,26.089130,29.146194,6.307213
1,2025-06-02 00:00:00+00:00,25.623104,25.677082,6.363144
2,2025-06-03 00:00:00+00:00,25.497006,25.700653,6.483844
3,2025-06-04 00:00:00+00:00,26.575173,24.377693,5.448498
4,2025-06-05 00:00:00+00:00,25.582770,26.363271,5.588969


## Benchmark info table

In [6]:
benchmark_info = pd.DataFrame([
    dict(source="Sentinel-1 RTC", resolution="10-20 m", revisit="~6 days", latency="hours-1 day",
         known_limitation="layover/shadow invalidates ~5% of pixels on sloped terrain",
         governing_record="docs/decisions/data-access-layer.md"),
    dict(source="Sentinel-2 L2A", resolution="10 m", revisit="~5 days", latency="hours",
         known_limitation="cloud cover, ~35% simulated gap rate for the pilot region",
         governing_record="docs/decisions/data-access-layer.md"),
    dict(source="Landsat-8/9", resolution="30 m", revisit="~8-16 days", latency="hours-1 day",
         known_limitation="no decision record yet — see 01_data_explorer.ipynb summary",
         governing_record="MISSING — gap identified"),
    dict(source="SMAP", resolution="~9 km", revisit="~2-3 days", latency="~2-3 days",
         known_limitation="coarse footprint, regional not zone-level signal",
         governing_record="MISSING — gap identified"),
    dict(source="ISMN", resolution="point", revisit="sub-daily-daily", latency="days-weeks (QC)",
         known_limitation="sparse station coverage in target region, unverified",
         governing_record="MISSING — gap identified"),
    dict(source="SoilGrids", resolution="250 m", revisit="static", latency="n/a",
         known_limitation="static — does not capture within-season change",
         governing_record="docs/decisions/data-access-layer.md"),
    dict(source="Open-Meteo", resolution="~10-25 km grid", revisit="hourly", latency="near-real-time",
         known_limitation="operational forecast accuracy, not a ground truth",
         governing_record="docs/decisions/weather-forcing-split.md"),
    dict(source="ERA5-Land", resolution="~9 km (0.1deg)", revisit="hourly", latency="~2-3 months",
         known_limitation="MUST NOT be used in the operational path (publication lag)",
         governing_record="docs/decisions/weather-forcing-split.md"),
])
benchmark_info


,source,resolution,revisit,latency,known_limitation,governing_record
0,Sentinel-1 RTC,10-20 m,~6 days,hours-1 day,layover/shadow invalidates ~5% of pixels on sl...,docs/decisions/data-access-layer.md
1,Sentinel-2 L2A,10 m,~5 days,hours,"cloud cover, ~35% simulated gap rate for the p...",docs/decisions/data-access-layer.md
2,Landsat-8/9,30 m,~8-16 days,hours-1 day,no decision record yet — see 01_data_explorer....,MISSING — gap identified
3,SMAP,~9 km,~2-3 days,~2-3 days,"coarse footprint, regional not zone-level signal",MISSING — gap identified
4,ISMN,point,sub-daily-daily,days-weeks (QC),"sparse station coverage in target region, unve...",MISSING — gap identified
5,SoilGrids,250 m,static,n/a,static — does not capture within-season change,docs/decisions/data-access-layer.md
6,Open-Meteo,~10-25 km grid,hourly,near-real-time,"operational forecast accuracy, not a ground truth",docs/decisions/weather-forcing-split.md
7,ERA5-Land,~9 km (0.1deg),hourly,~2-3 months,MUST NOT be used in the operational path (publ...,docs/decisions/weather-forcing-split.md
